## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [30]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
# Use as many cells as you like

In [31]:
nasdaq100_ca = pd.read_csv('nasdaq100_CA.csv')
price_change = pd.read_csv('nasdaq100_price_change.csv')

nasdaq100_ca = nasdaq100_ca.merge(price_change[['symbol', 'ytd']], on='symbol')
nasdaq100_ca.head()

,symbol,name,headQuarter,dateFirstAdded,cik,founded,ytd
0,AAPL,Apple Inc.,"Cupertino, CA",NaN,320193,1976-04-01,42.99992
1,ABNB,Airbnb,"San Francisco, CA",NaN,1559720,2008-08-01,68.66902
2,ADBE,Adobe Inc.,"San Jose, CA",NaN,796343,1982-12-01,57.22723
3,ADSK,Autodesk,"San Rafael, CA",NaN,769397,1982-01-30,10.02701
4,AMAT,Applied Materials,"Santa Clara, CA",NaN,6951,1967-11-10,55.46366


In [32]:
symbols_list = nasdaq100_ca['symbol'].tolist()

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": f"""Classify each of these stock symbols into one of these sectors: 
        Technology, Consumer Cyclical, Industrials, Utilities, Healthcare, Communication, Energy, Consumer Defensive, Real Estate, Financial.
        Return only a Python dictionary where keys are symbols and values are sectors, no extra text.
        Symbols: {symbols_list}"""}
    ]
)

import ast
sector_dict = ast.literal_eval(response.choices[0].message.content)
nasdaq100_ca['sector'] = nasdaq100_ca['symbol'].map(sector_dict)
nasdaq = nasdaq100_ca.copy()
print(nasdaq100_ca.columns)
nasdaq100_ca.head()

Index(['symbol', 'name', 'headQuarter', 'dateFirstAdded', 'cik', 'founded',
       'ytd', 'sector'],
      dtype='object')


,symbol,name,headQuarter,dateFirstAdded,cik,founded,ytd,sector
0,AAPL,Apple Inc.,"Cupertino, CA",NaN,320193,1976-04-01,42.99992,Technology
1,ABNB,Airbnb,"San Francisco, CA",NaN,1559720,2008-08-01,68.66902,Technology
2,ADBE,Adobe Inc.,"San Jose, CA",NaN,796343,1982-12-01,57.22723,Technology
3,ADSK,Autodesk,"San Rafael, CA",NaN,769397,1982-01-30,10.02701,Technology
4,AMAT,Applied Materials,"Santa Clara, CA",NaN,6951,1967-11-10,55.46366,Technology


In [33]:
nasdaq_info = nasdaq[['symbol', 'sector', 'ytd']].to_string()

response2 = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": f"""Based on this Nasdaq stock data with YTD performance:
        {nasdaq_info}
        
        Recommend the two best sectors and at least two companies per sector.
        Explain why each sector and company is a good pick."""}
    ]
)

stock_recommendations = response2.choices[0].message.content
print(stock_recommendations)

Based on the YTD performance data provided, the two best sectors to consider investing in are Technology and Communication.

1. Technology Sector:
a. NVIDIA (NVDA) - YTD Performance: 217.26511
   - NVIDIA is a leading technology company known for its graphics processing units (GPUs) used in gaming, artificial intelligence, and data centers. The company has seen significant growth in its stock price year-to-date, reflecting strong market demand for its products and services.

b. Meta Platforms (META) - YTD Performance: 153.77585
   - Formerly known as Facebook, Meta Platforms is a dominant player in the social media and technology industry. The company has demonstrated impressive performance in 2022, driven by its diverse revenue streams, including advertising and virtual reality products.

2. Communication Sector:
a. Airbnb (ABNB) - YTD Performance: 68.66902
   - Airbnb is a technology company that operates an online marketplace for lodging, primarily homestays for vacation rentals and